<a href="https://colab.research.google.com/github/ithelga/beeline-banner-ab-test/blob/dev/notebooks/Team1_HW5_CDA_i1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SmartAds Efficiency**: Оптимизация эффективности маркетинговых каналов

## [STAGE 4_1] **Проведение подтверждающего анализа (СDA) - 1-ая итерация**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import warnings
import polars as pl
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

graph_path = 'drive/MyDrive/Colab Notebooks/Beeline Banner AB-test/graph'
data_path = 'drive/MyDrive/Colab Notebooks/Beeline Banner AB-test/data'

Mounted at /content/drive


### Загружаем данные с Google Disk

In [ ]:
supertable = pd.read_csv(f'{data_path}/supertable.csv')
supertable.head()

,user_id,timestamp,date,shows,clicks,banner_first,campaign_first,placement,device_type,os,geo,has_install_event,has_first_order_event,has_registration_event,has_tariff_switch_event,segment,tariff,creative_type,size,target_audience_segment
0,1,2025-02-10 10:06:23,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,premium,premium,no_impression,no_impression,no_impression
1,1,2025-02-15 20:05:59,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,premium,premium,no_impression,no_impression,no_impression
2,1,2025-02-16 08:34:35,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,premium,premium,no_impression,no_impression,no_impression
3,1,2025-02-17 21:55:14,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,premium,premium,no_impression,no_impression,no_impression
4,1,2025-03-01 20:13:19,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,premium,premium,no_impression,no_impression,no_impression


### Проверка гипотезы 1

> Показ анимированного баннера дает CTR более чем на 25% выше по сравнению со статичным баннером.

####ШАГ 1. Проверка достаточности данных

In [ ]:
# ------------------------------------------------------------
# ШАГ 1. Подготовка данных для проверки гипотезы 1
# ------------------------------------------------------------

# Посмотрим распределение типов креативов
print("Распределение creative_type:\n")
display(supertable["creative_type"].value_counts())

# Агрегация показов и кликов по типу креатива
agg = (
    supertable.groupby("creative_type")
    .agg(
        shows=("shows", "sum"),
        clicks=("clicks", "sum")
    )
    .reset_index()
)

agg["ctr"] = agg["clicks"] / agg["shows"]
print("\nАгрегированная статистика по креативам:\n")
display(agg)

# ------------------------------------------------------------
# ШАГ 1.1. Формируем две группы:
# Static = статичные баннеры
# Animated = объединение video + rich
# ------------------------------------------------------------

static = agg[agg["creative_type"] == "static"]
animated = agg[agg["creative_type"].isin(["video", "rich"])]

print("\nСтатистика для STATIC:")
display(static)

print("\nСтатистика для ANIMATED (video + rich):")
display(animated)

# ------------------------------------------------------------
# ШАГ 1.2. Проверка достаточности данных для применения z-test
# Требования:
#   - кликов в группе >= 5
#   - некликов (shows - clicks) >= 5
# ------------------------------------------------------------

def check_group(shows, clicks, name):
    failures = shows - clicks
    ctr = clicks / shows if shows > 0 else None
    ok = (clicks >= 5) and (failures >= 5)

    print(f"--- {name} ---")
    print(f"Shows:     {shows}")
    print(f"Clicks:    {clicks}")
    print(f"Failures:  {failures}")
    print(f"CTR:       {ctr:.5f}")
    print(f"OK for z-test: {ok}\n")

    return ok

print("\nПроверка достаточности данных:\n")

ok_static = check_group(
    static["shows"].sum(),
    static["clicks"].sum(),
    "STATIC"
)

ok_animated = check_group(
    animated["shows"].sum(),
    animated["clicks"].sum(),
    "ANIMATED"
)

# Итоговый флаг пригодности данных
data_ok = ok_static and ok_animated

print("ИТОГОВЫЙ ВЕРДИКТ ПО ДОСТАТОЧНОСТИ ДАННЫХ:")
if data_ok:
    print("✔ Данных достаточно — можно использовать z-test для двух долей.")
else:
    print("✘ Данных недостаточно — потребуется точный тест Фишера.")

Распределение creative_type:



,count
creative_type,
no_impression,1170267
static,859
video,614
rich,108



Агрегированная статистика по креативам:



,creative_type,shows,clicks,ctr
0,no_impression,0,0,NaN
1,rich,124,10,0.080645
2,static,974,76,0.078029
3,video,717,71,0.099024



Статистика для STATIC:


,creative_type,shows,clicks,ctr
2,static,974,76,0.078029



Статистика для ANIMATED (video + rich):


,creative_type,shows,clicks,ctr
1,rich,124,10,0.080645
3,video,717,71,0.099024



Проверка достаточности данных:

--- STATIC ---
Shows:     974
Clicks:    76
Failures:  898
CTR:       0.07803
OK for z-test: True

--- ANIMATED ---
Shows:     841
Clicks:    81
Failures:  760
CTR:       0.09631
OK for z-test: True

ИТОГОВЫЙ ВЕРДИКТ ПО ДОСТАТОЧНОСТИ ДАННЫХ:
✔ Данных достаточно — можно использовать z-test для двух долей.


Для проверки гипотезы 1 были выделены две группы креативов: ***Static*** (статичные баннеры) и ***Animated*** *(объединённые форматы video и rich)*. По каждой группе рассчитано *количество показов, кликов и CTR.*

Проверка минимальных требований для применения ***z-test*** показала:

*   группа ***Static*** содержит 974 показов и 76 кликов (CTR = 7.8%),
*   группа ***Animated*** содержит 841 показ и 81 клик (CTR = 9.6%).

В обеих группах количество кликов превышает минимально необходимый порог (≥ 5), как и количество неуспешных исходов (shows – clicks ≥ 5).

Таким образом:
1.   Данные являются **достаточными** для применения ***z-test*** для двух пропорций.
2.   Альтернативные точные методы (например, Fisher exact test) не требуются.


####ШАГ 2. Проверка гипотезы

In [ ]:
# Достаём агрегацию
shows_static = static["shows"].sum()
clicks_static = static["clicks"].sum()

shows_anim = animated["shows"].sum()
clicks_anim = animated["clicks"].sum()

# Фактические CTR
p_static = clicks_static / shows_static
p_anim = clicks_anim / shows_anim
lift = p_anim / p_static - 1

print("Фактические значения:")
print(f"CTR Static:    {p_static:.5f}")
print(f"CTR Animated:  {p_anim:.5f}")
print(f"Lift:          {lift:.5f}  (~{lift*100:.2f}%)\n")


# ------------------------------------------------------------
# 2.1 Z-test для двух долей
# Проверяем H1: CTR_anim > CTR_static
# ------------------------------------------------------------

stat, p_value = proportions_ztest(
    count=[clicks_anim, clicks_static],
    nobs=[shows_anim, shows_static],
    alternative='larger'
)

print("Z-test результаты:")
print(f"Z-statistic: {stat:.4f}")
print(f"P-value:     {p_value:.6f}\n")


# ------------------------------------------------------------
# 2.2 Доверительные интервалы
# CI для каждой доли + CI для разницы и CI для лифта
# ------------------------------------------------------------

# CI для отдельных CTR
ci_static = proportion_confint(clicks_static, shows_static, method="normal")
ci_anim = proportion_confint(clicks_anim, shows_anim, method="normal")

# CI для разницы долей
diff = p_anim - p_static
se = np.sqrt(p_anim*(1 - p_anim)/shows_anim + p_static*(1 - p_static)/shows_static)
z = 1.96  # 95% CI
diff_low, diff_high = diff - z*se, diff + z*se

# CI для лифта (приближение через разницу)
lift_low = diff_low / p_static
lift_high = diff_high / p_static

print("Доверительные интервалы:")
print(f"CI CTR Static:         ({ci_static[0]:.5f}, {ci_static[1]:.5f})")
print(f"CI CTR Animated:       ({ci_anim[0]:.5f}, {ci_anim[1]:.5f})")
print(f"CI разницы CTR (anim - static): ({diff_low:.5f}, {diff_high:.5f})")
print(f"CI лифта:              ({lift_low:.5f}, {lift_high:.5f})\n")


# ------------------------------------------------------------
# 2.3 Проверка порога: превышает ли нижняя граница CI > 0.25?
# ------------------------------------------------------------

threshold = 0.25
passed = lift_low > threshold

print("Проверка порогового условия:")
print(f"Требуемый порог LIFT > 25%")
print(f"Фактический LIFT:      {lift:.5f}  (~{lift*100:.2f}%)")
print(f"Нижняя граница CI:     {lift_low:.5f}")

if passed:
    print("✔ Гипотеза подтверждается на уровне CDA.")
else:
    print("✘ Гипотеза НЕ подтверждается на уровне CDA.")

Фактические значения:
CTR Static:    0.07803
CTR Animated:  0.09631
Lift:          0.23434  (~23.43%)

Z-test результаты:
Z-statistic: 1.3819
P-value:     0.083503

Доверительные интервалы:
CI CTR Static:         (0.06118, 0.09487)
CI CTR Animated:       (0.07637, 0.11625)
CI разницы CTR (anim - static): (-0.00782, 0.04439)
CI лифта:              (-0.10018, 0.56886)

Проверка порогового условия:
Требуемый порог LIFT > 25%
Фактический LIFT:      0.23434  (~23.43%)
Нижняя граница CI:     -0.10018
✘ Гипотеза НЕ подтверждается на уровне CDA.


Мы сравнили эффективность двух типов баннеров — статичных и анимированных (video + rich).

**Результаты:**
*   CTR статичных баннеров: 7.80%
*   CTR анимированных баннеров: 9.63%
*   Прирост CTR: +23.4%

То есть анимированные баннеры дейтвительно демонстрируют более высокий CTR

Однако статистическая проверка показала, что:

1. Различие между группами не является статистически значимым (p = 0.0835)
2. Доверительный интервал для эффекта широкий — он включает даже отрицательные значения
3. Нижняя граница интервала не достигает порога +25%, указанного в гипотезе.

Это означает, что наблюдаемый рост CTR может быть частично обусловлен случайными колебаниями данных, и у нас нет достаточных оснований утверждать, что анимированные баннеры стабильно превосходят статичные на 25% или больше.

**Итог: Гипотеза 1 не подтверждается на этапе CDA.**

### Проверка гипотезы 2

> Пользователи веб-версии не менее, чем на 30% чаще кликают на рекламу (CTR), чем пользователи мобильного приложения

####ШАГ 1. Проверка достаточности данных

In [ ]:
# ------------------------------------------------------------
# ШАГ 1. Подготовка данных для проверки гипотезы 2
# ------------------------------------------------------------

print("Распределение os:\n")
display(supertable["os"].value_counts())

# Агрегация показов и кликов по OS
os_stats = (
    supertable.groupby("os")
    .agg(
        shows=("shows", "sum"),
        clicks=("clicks", "sum")
    )
    .reset_index()
)

os_stats["ctr"] = os_stats["clicks"] / os_stats["shows"]

print("\nАгрегированная статистика по OS:\n")
display(os_stats)

# ------------------------------------------------------------
# ШАГ 1.1. Формируем две группы:
# WEB = web
# MOBILE = android + ios
# ------------------------------------------------------------

web = os_stats[os_stats["os"] == "web"]

android = os_stats[os_stats["os"] == "android"]
ios = os_stats[os_stats["os"] == "ios"]

shows_web = web["shows"].sum()
clicks_web = web["clicks"].sum()

shows_mobile = android["shows"].sum() + ios["shows"].sum()
clicks_mobile = android["clicks"].sum() + ios["clicks"].sum()

print("\nСтатистика для WEB:")
display(web)

print("\nСтатистика для MOBILE (android + ios):")
print(f"Shows:  {shows_mobile}")
print(f"Clicks: {clicks_mobile}")
print(f"CTR:    {clicks_mobile / shows_mobile:.5f}")

# ------------------------------------------------------------
# ШАГ 1.2. Проверка достаточности данных для применения z-test
# Требования:
#   - кликов ≥ 5
#   - некликов ≥ 5
# ------------------------------------------------------------

def check_group(shows, clicks, name):
    failures = shows - clicks
    ctr = clicks / shows
    ok = (clicks >= 5) and (failures >= 5)

    print(f"\n--- {name} ---")
    print(f"Shows:     {shows}")
    print(f"Clicks:    {clicks}")
    print(f"Failures:  {failures}")
    print(f"CTR:       {ctr:.5f}")
    print(f"OK for z-test: {ok}")

    return ok

print("\nПроверка достаточности данных:\n")

ok_web = check_group(shows_web, clicks_web, "WEB")
ok_mobile = check_group(shows_mobile, clicks_mobile, "MOBILE")

data_ok = ok_web and ok_mobile

print("\nИТОГОВЫЙ ВЕРДИКТ ПО ДОСТАТОЧНОСТИ ДАННЫХ:")
if data_ok:
    print("✔ Данных достаточно — можно использовать z-test для двух долей.")
else:
    print("✘ Данных недостаточно — потребуется точный тест Фишера.")

Распределение os:



,count
os,
ios,743
android,707
web,131



Агрегированная статистика по OS:



,os,shows,clicks,ctr
0,android,763,67,0.087811
1,ios,920,75,0.081522
2,web,132,15,0.113636



Статистика для WEB:


,os,shows,clicks,ctr
2,web,132,15,0.113636



Статистика для MOBILE (android + ios):
Shows:  1683
Clicks: 142
CTR:    0.08437

Проверка достаточности данных:


--- WEB ---
Shows:     132
Clicks:    15
Failures:  117
CTR:       0.11364
OK for z-test: True

--- MOBILE ---
Shows:     1683
Clicks:    142
Failures:  1541
CTR:       0.08437
OK for z-test: True

ИТОГОВЫЙ ВЕРДИКТ ПО ДОСТАТОЧНОСТИ ДАННЫХ:
✔ Данных достаточно — можно использовать z-test для двух долей.


Для проверки гипотезы 2 были выделены две группы пользователей:
***WEB*** *(пользователи веб-версии)* и ***MOBILE*** *(объединённые платформы Android и iOS).*
По каждой группе рассчитано количество показов, кликов и CTR.

Проверка минимальных требований для применения z-test показала:
*   группа ***WEB*** содержит 132 показа и 15 кликов (CTR = 11.36%),
*   группа ***MOBILE*** содержит 1683 показа и 142 клика (CTR = 8.44%).

В обеих группах количество кликов превышает минимально необходимый порог (≥ 5), как и количество неуспешных исходов (shows – clicks ≥ 5).

Таким образом:
1.   Данные являются **достаточными** для применения ***z-test*** для двух пропорций.
2.   Альтернативные точные методы (например, Fisher exact test) не требуются.





####ШАГ 2. Проверка гипотезы

In [ ]:
# ------------------------------------------------------------
# ШАГ 2. Проверка гипотезы 2
# ------------------------------------------------------------

# Фактические CTR
p_web = clicks_web / shows_web
p_mobile = clicks_mobile / shows_mobile
lift = p_web / p_mobile - 1

print("Фактические значения:")
print(f"CTR WEB:       {p_web:.5f}")
print(f"CTR MOBILE:    {p_mobile:.5f}")
print(f"Lift:          {lift:.5f} (~{lift*100:.2f}%)\n")


# ------------------------------------------------------------
# 2.1. Z-test: проверяем, кликают ли пользователи WEB чаще MOBILE
# ------------------------------------------------------------

stat, p_value = proportions_ztest(
    count=[clicks_web, clicks_mobile],
    nobs=[shows_web, shows_mobile],
    alternative="larger"   # проверяем CTR_web > CTR_mobile
)

print("Результаты Z-test:")
print(f"Z-statistic: {stat:.4f}")
print(f"P-value:     {p_value:.6f}\n")


# ------------------------------------------------------------
# 2.2. Доверительные интервалы
# ------------------------------------------------------------

# CI для CTR
ci_web = proportion_confint(clicks_web, shows_web, method="normal")
ci_mobile = proportion_confint(clicks_mobile, shows_mobile, method="normal")

# CI для разницы CTR
diff = p_web - p_mobile
se = np.sqrt(p_web*(1-p_web)/shows_web + p_mobile*(1-p_mobile)/shows_mobile)
z = 1.96
diff_low, diff_high = diff - z*se, diff + z*se

# CI для лифта
lift_low = diff_low / p_mobile
lift_high = diff_high / p_mobile

print("Доверительные интервалы:")
print(f"CI CTR WEB:       ({ci_web[0]:.5f}, {ci_web[1]:.5f})")
print(f"CI CTR MOBILE:    ({ci_mobile[0]:.5f}, {ci_mobile[1]:.5f})")
print(f"CI diff CTR:      ({diff_low:.5f}, {diff_high:.5f})")
print(f"CI lift:          ({lift_low:.5f}, {lift_high:.5f})\n")


# ------------------------------------------------------------
# 2.3 Проверка порогового условия: lift >= 30%
# ------------------------------------------------------------

threshold = 0.30
print("Порог гипотезы: 0.30 (30%)")

if lift_low > threshold:
    print("✔ Гипотеза подтверждается.")
else:
    print("✘ Гипотеза НЕ подтверждается.")

Фактические значения:
CTR WEB:       0.11364
CTR MOBILE:    0.08437
Lift:          0.34683 (~34.68%)

Результаты Z-test:
Z-statistic: 1.1517
P-value:     0.124718

Доверительные интервалы:
CI CTR WEB:       (0.05950, 0.16778)
CI CTR MOBILE:    (0.07109, 0.09765)
CI diff CTR:      (-0.02648, 0.08501)
CI lift:          (-0.31388, 1.00755)

Порог гипотезы: 0.30 (30%)
✘ Гипотеза НЕ подтверждается.


Мы сравнили эффективность двух групп пользователей — веб-версии (WEB) и мобильного приложения (MOBILE: Android + iOS)

**Результаты:**

*   CTR WEB: 11.36%
*   CTR MOBILE: 8.44%
*   Прирост CTR (lift): +34.68%

То есть веб-пользователи действительно демонстрируют более высокий CTR на уровне фактических данных.

Однако статистическая проверка показала, что:

1. Различие между группами не является статистически значимым (p = 0.1247).
2. Доверительный интервал для эффекта очень широкий — он варьируется от отрицательных значений до более чем 100% прироста.
3. Нижняя граница доверительного интервала для лифта не достигает порога +30%, заложенного в гипотезе (фактически она составляет –31%).

Это означает, что наблюдаемый рост CTR у веб-пользователей может быть частично обусловлен случайностью, и текущего объёма данных недостаточно, чтобы уверенно утверждать, что веб стабильно превосходит mobile по CTR на 30% или больше.

**Итог: Гипотеза 2 не подтверждается на этапе CDA.**

### Проверка гипотезы 3

> Размещение рекламы в приложении увеличивает вероятность смены тарифа не менее чем на 25% по сравнению с сайтом/соц сетями

####ШАГ 1. Проверка достаточности данных

In [ ]:
# ------------------------------------------------------------
# ШАГ 1. Подготовка данных для проверки гипотезы 3
# ------------------------------------------------------------

print("Распределение placement:\n")
display(supertable["placement"].value_counts())

# Агрегируем показы и количество пользователей со сменой тарифа
placement_stats = (
    supertable.groupby("placement")
    .agg(
        shows=("shows", "sum"),
        switches=("has_tariff_switch_event", "sum")
    )
    .reset_index()
)

placement_stats["cr"] = placement_stats["switches"] / placement_stats["shows"]

print("\nАгрегированная статистика по placement:\n")
display(placement_stats)

# Выделяем группы
app = placement_stats[placement_stats["placement"] == "app"]
site = placement_stats[placement_stats["placement"] == "site"]
social = placement_stats[placement_stats["placement"] == "social"]

# Статистика MOBILE (две группы сравнения)
print("\nСтатистика для APP:")
display(app)

print("\nСтатистика для SITE:")
display(site)

print("\nСтатистика для SOCIAL:")
display(social)

# ------------------------------------------------------------
# ШАГ 1.2. Проверка достаточности данных для применения z-test
# Требование:
#   - successes >= 5 (switch events)
#   - failures >= 5  (shows - switches)
# ------------------------------------------------------------

def check_group(shows, switches, name):
    failures = shows - switches
    cr = switches / shows
    ok = (switches >= 5) and (failures >= 5)

    print(f"\n--- {name} ---")
    print(f"Shows:       {shows}")
    print(f"Switches:    {switches}")
    print(f"Failures:    {failures}")
    print(f"Conversion:  {cr:.5f}")
    print(f"OK for z-test: {ok}")

    return ok

print("\nПроверка достаточности данных:\n")

ok_app = check_group(app["shows"].sum(), app["switches"].sum(), "APP")
ok_site = check_group(site["shows"].sum(), site["switches"].sum(), "SITE")
ok_social = check_group(social["shows"].sum(), social["switches"].sum(), "SOCIAL")

data_ok = ok_app and ok_site and ok_social

print("\nИТОГОВЫЙ ВЕРДИКТ ПО ДОСТАТОЧНОСТИ ДАННЫХ:")
if data_ok:
    print("✔ Данных достаточно — можно применять z-test для пропорций.")
else:
    print("✘ Данных недостаточно — потребуется точный тест Фишера.")

Распределение placement:



,count
placement,
site,640
app,608
social,333



Агрегированная статистика по placement:



,placement,shows,switches,cr
0,app,679,96,0.141384
1,site,744,81,0.108871
2,social,392,44,0.112245



Статистика для APP:


...


Статистика для SITE:


,placement,shows,switches,cr
1,site,744,81,0.108871



Статистика для SOCIAL:


,placement,shows,switches,cr
2,social,392,44,0.112245



Проверка достаточности данных:


--- APP ---
Shows:       679
Switches:    96
Failures:    583
Conversion:  0.14138
OK for z-test: True

--- SITE ---
Shows:       744
Switches:    81
Failures:    663
Conversion:  0.10887
OK for z-test: True

--- SOCIAL ---
Shows:       392
Switches:    44
Failures:    348
Conversion:  0.11224
OK for z-test: True

ИТОГОВЫЙ ВЕРДИКТ ПО ДОСТАТОЧНОСТИ ДАННЫХ:
✔ Данных достаточно — можно применять z-test для пропорций.


Для проверки гипотезы 3 были выделены три группы размещения рекламы:
***APP*** *(реклама внутри приложения)*, ***SITE*** *(размещение на сайте)* и ***SOCIAL*** *(размещение в социальных сетях)*.
По каждой группе рассчитано количество показов, количество пользователей со сменой тарифа и конверсия показ → смена тарифа (CR).

Проверка минимальных требований для применения z-test показала:

*   группа ***APP*** содержит 679 показов и 96 случаев смены тарифа (CR = 14.14%),
*   группа ***SITE*** содержит 744 показа и 81 смену тарифа (CR = 10.89%),
*   группа ***SOCIAL*** содержит 392 показа и 44 смены тарифа (CR = 11.22%).


Во всех группах количество “успешных” наблюдений (switches ≥ 5) и “неуспешных” (shows – switches ≥ 5) достаточно для корректного применения статистического критерия.

Таким образом:
1.   Данные являются достаточными для применения z-test для сравнения пропорций.
2.   Использование альтернативных точных тестов (например, Fisher exact test) не требуется.

####ШАГ 2. Проверка гипотезы

In [ ]:
# ------------------------------------------------------------
# ШАГ 2. Проверка гипотезы 3
# ------------------------------------------------------------

# Фактические CR
p_app = app["switches"].sum() / app["shows"].sum()
p_site = site["switches"].sum() / site["shows"].sum()
p_social = social["switches"].sum() / social["shows"].sum()

shows_app, switches_app = app["shows"].sum(), app["switches"].sum()
shows_site, switches_site = site["shows"].sum(), site["switches"].sum()
shows_social, switches_social = social["shows"].sum(), social["switches"].sum()

lift_app_site = p_app / p_site - 1
lift_app_social = p_app / p_social - 1

print("Фактические значения:")
print(f"CR APP:     {p_app:.5f}")
print(f"CR SITE:    {p_site:.5f}")
print(f"CR SOCIAL:  {p_social:.5f}")
print(f"Lift APP > SITE:    {lift_app_site:.5f} (~{lift_app_site*100:.2f}%)")
print(f"Lift APP > SOCIAL:  {lift_app_social:.5f} (~{lift_app_social*100:.2f}%)\n")


# ------------------------------------------------------------
# 2.1. Z-test APP vs SITE
# ------------------------------------------------------------

stat_as, p_as = proportions_ztest(
    count=[switches_app, switches_site],
    nobs=[shows_app, shows_site],
    alternative="larger"  # CR_app > CR_site
)

# ------------------------------------------------------------
# 2.2. Z-test APP vs SOCIAL
# ------------------------------------------------------------

stat_aso, p_aso = proportions_ztest(
    count=[switches_app, switches_social],
    nobs=[shows_app, shows_social],
    alternative="larger"  # CR_app > CR_social
)

print("Результаты Z-test:")
print(f"APP > SITE:   Z = {stat_as:.4f},  p = {p_as:.6f}")
print(f"APP > SOCIAL: Z = {stat_aso:.4f}, p = {p_aso:.6f}\n")


# ------------------------------------------------------------
# 2.3. Доверительные интервалы для лифтов
# ------------------------------------------------------------

def compute_ci_lift(p1, p2, n1, n2):
    diff = p1 - p2
    se = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    z = 1.96
    diff_low, diff_high = diff - z*se, diff + z*se
    lift_low = diff_low / p2
    lift_high = diff_high / p2
    return lift_low, lift_high

ci_lift_app_site = compute_ci_lift(p_app, p_site, shows_app, shows_site)
ci_lift_app_social = compute_ci_lift(p_app, p_social, shows_app, shows_social)

print("Доверительные интервалы для лифта:")
print(f"CI APP > SITE:   ({ci_lift_app_site[0]:.5f}, {ci_lift_app_site[1]:.5f})")
print(f"CI APP > SOCIAL: ({ci_lift_app_social[0]:.5f}, {ci_lift_app_social[1]:.5f})\n")


# ------------------------------------------------------------
# 2.4. Проверка порогового условия
# ------------------------------------------------------------

threshold = 0.25
print("Порог гипотезы: 0.25 (25%)")

passed_site = ci_lift_app_site[0] > threshold
passed_social = ci_lift_app_social[0] > threshold

print("\nИТОГОВОЕ РЕШЕНИЕ:")
if passed_site and passed_social:
    print("✔ Гипотеза подтверждается для обеих групп одновременно.")
else:
    print("✘ Гипотеза НЕ подтверждается.")


Фактические значения:
CR APP:     0.14138
CR SITE:    0.10887
CR SOCIAL:  0.11224
Lift APP > SITE:    0.29864 (~29.86%)
Lift APP > SOCIAL:  0.25961 (~25.96%)

Результаты Z-test:
APP > SITE:   Z = 1.8563,  p = 0.031708
APP > SOCIAL: Z = 1.3627, p = 0.086481

Доверительные интервалы для лифта:
CI APP > SITE:   (-0.01792, 0.61520)
CI APP > SOCIAL: (-0.10374, 0.62296)

Порог гипотезы: 0.25 (25%)

ИТОГОВОЕ РЕШЕНИЕ:
✘ Гипотеза НЕ подтверждается.


Мы сравнили эффективность трёх каналов размещения рекламы — APP (приложение), SITE (сайт) и SOCIAL (социальные сети)

Результаты:


*   CR APP: 14.14%
*   CR SITE: 10.89%
*   CR SOCIAL: 11.22%
*   Прирост APP > SITE: +29.86%
*   Прирост APP > SOCIAL: +25.96%

То есть на уровне фактических данных размещение в приложении действительно показывает более высокую конверсию в смену тарифа по сравнению с сайтом и соцсетями.

Однако статистическая проверка показала, что:



1.   Различие между группами не является устойчивым одновременно для обеих пар сравнений.
  *   Для APP > SITE: p = 0.0317 → статистически значимо
  *   Для APP > SOCIAL: p = 0.0865 → **не значимо**
2.   Доверительные интервалы для обоих лифтов очень широкие и включают отрицательные значения, что указывает на высокую неопределённость в оценке эффекта.
  *   CI(APP > SITE): от –1.79% до +61.52%
  *   CI(APP > SOCIAL): от –10.37% до +62.30%
3. Нижняя граница доверительных интервалов в обоих сравнениях не достигает порога +25%, требуемого гипотезой (в одном случае она отрицательная, в другом — также отрицательная).

Это означает, что наблюдаемое повышение вероятности смены тарифа в приложении может быть частично обусловлено случайными колебаниями, и данных недостаточно, чтобы уверенно утверждать, что эффект стабилен и действительно превышает 25% по сравнению с сайтом и соцсетями.

**Итог: Гипотеза 3 не подтверждается на этапе CDA.**

### Проверка гипотезы 4

> Просмотр рекламы на устройстве tablet увеличивает вероятность смены тарифа более, чем на 10 %  по сравнению с другими устройствами после регистрации пользователей

####ШАГ 1. Проверка достаточности данных

In [ ]:
# ------------------------------------------------------------
# ШАГ 1. Подготовка данных для проверки гипотезы 4
# ------------------------------------------------------------

print("Распределение device_type:\n")
display(supertable["device_type"].value_counts())

# Агрегация по типу устройства
device_stats = (
    supertable.groupby("device_type")
    .agg(
        registrations=("has_registration_event", "sum"),
        switches=("has_tariff_switch_event", "sum")
    )
    .reset_index()
)

device_stats["cr"] = device_stats["switches"] / device_stats["registrations"]

print("\nАгрегированная статистика по device_type:\n")
display(device_stats)

# ------------------------------------------------------------
# Формируем две группы:
# TABLET vs PHONE (если есть другие устройства — они игнорируются)
# ------------------------------------------------------------

tablet = device_stats[device_stats["device_type"] == "tablet"]
phone = device_stats[device_stats["device_type"] == "phone"]

shows_tablet = tablet["registrations"].sum()
switches_tablet = tablet["switches"].sum()

shows_phone = phone["registrations"].sum()
switches_phone = phone["switches"].sum()

print("\nСтатистика для TABLET:")
display(tablet)

print("\nСтатистика для PHONE:")
display(phone)

# ------------------------------------------------------------
# ШАГ 1.2. Проверка достаточности данных
# Требования для z-test:
# - switches >= 5
# - registrations - switches >= 5
# ------------------------------------------------------------

def check_group(registrations, switches, name):
    failures = registrations - switches
    cr = switches / registrations if registrations > 0 else None
    ok = (switches >= 5) and (failures >= 5)

    print(f"\n--- {name} ---")
    print(f"Registrations: {registrations}")
    print(f"Switches:      {switches}")
    print(f"Failures:      {failures}")
    print(f"CR:            {cr:.5f}")
    print(f"OK for z-test: {ok}")

    return ok

print("\nПроверка достаточности данных:")

ok_tablet = check_group(shows_tablet, switches_tablet, "TABLET")
ok_phone = check_group(shows_phone, switches_phone, "PHONE")

data_ok = ok_tablet and ok_phone

print("\nИТОГОВЫЙ ВЕРДИКТ ПО ДОСТАТОЧНОСТИ ДАННЫХ:")
if data_ok:
    print("✔ Данных достаточно — можно применять z-test.")
else:
    print("✘ Данных недостаточно — потребуется точный тест Фишера.")


Распределение device_type:



,count
device_type,
phone,1332
tablet,249



Агрегированная статистика по device_type:



,device_type,registrations,switches,cr
0,phone,683,184,0.269400
1,tablet,123,37,0.300813



Статистика для TABLET:


,device_type,registrations,switches,cr
1,tablet,123,37,0.300813



Статистика для PHONE:


,device_type,registrations,switches,cr
0,phone,683,184,0.2694



Проверка достаточности данных:

--- TABLET ---
Registrations: 123
Switches:      37
Failures:      86
CR:            0.30081
OK for z-test: True

--- PHONE ---
Registrations: 683
Switches:      184
Failures:      499
CR:            0.26940
OK for z-test: True

ИТОГОВЫЙ ВЕРДИКТ ПО ДОСТАТОЧНОСТИ ДАННЫХ:
✔ Данных достаточно — можно применять z-test.


Для проверки гипотезы 4 были выделены две группы устройств ***TABLET*** и ***PHONE***. По каждой группе рассчитано количество пользователей с регистрацией, количество пользователей, сменивших тариф, и конверсия регистрация → смена тарифа (CR).

Проверка минимальных требований для применения z-test показала:

*   группа ***TABLET*** содержит 123 регистрации и 37 случаев смены тарифа
(CR = 30.08%)
*   группа ***PHONE*** содержит 683 регистрации и 184 случаев смены тарифа
(CR = 26.94%)

В обеих группах количество “успешных” наблюдений (switches ≥ 5) и “неуспешных” (registrations – switches ≥ 5) достаточно для корректного применения статистического критерия.

Таким образом:
1.   Данные являются **достаточными** для применения z-test для сравнения пропорций.
2.   Использование альтернативных точных тестов (например, Fisher exact test) не требуется.

####ШАГ 2. Проверка гипотезы

In [ ]:
# ------------------------------------------------------------
# ШАГ 2. Проверка гипотезы 4
# ------------------------------------------------------------

# Фактические конверсии
p_tablet = switches_tablet / shows_tablet
p_phone = switches_phone / shows_phone
lift = p_tablet / p_phone - 1

print("Фактические значения:")
print(f"CR TABLET:  {p_tablet:.5f}")
print(f"CR PHONE:   {p_phone:.5f}")
print(f"Lift:       {lift:.5f} (~{lift*100:.2f}%)\n")


# ------------------------------------------------------------
# 2.1. Z-test: проверяем CR_tablet > CR_phone
# ------------------------------------------------------------

stat, p_value = proportions_ztest(
    count=[switches_tablet, switches_phone],
    nobs=[shows_tablet, shows_phone],
    alternative="larger"   # проверяем CR_tablet > CR_phone
)

print("Результаты Z-test:")
print(f"Z-statistic: {stat:.4f}")
print(f"P-value:     {p_value:.6f}\n")


# ------------------------------------------------------------
# 2.2. Доверительные интервалы
# ------------------------------------------------------------

ci_tablet = proportion_confint(switches_tablet, shows_tablet, method="normal")
ci_phone = proportion_confint(switches_phone, shows_phone, method="normal")

diff = p_tablet - p_phone
se = np.sqrt(p_tablet*(1-p_tablet)/shows_tablet +
             p_phone*(1-p_phone)/shows_phone)

z = 1.96
diff_low, diff_high = diff - z*se, diff + z*se

lift_low = diff_low / p_phone
lift_high = diff_high / p_phone

print("Доверительные интервалы:")
print(f"CI CR TABLET: ({ci_tablet[0]:.5f}, {ci_tablet[1]:.5f})")
print(f"CI CR PHONE:  ({ci_phone[0]:.5f}, {ci_phone[1]:.5f})")
print(f"CI diff:      ({diff_low:.5f}, {diff_high:.5f})")
print(f"CI lift:      ({lift_low:.5f}, {lift_high:.5f})\n")


# ------------------------------------------------------------
# 2.3. Проверка порогового условия: lift ≥ 10%
# ------------------------------------------------------------

threshold = 0.10
print("Порог гипотезы: 0.10 (10%)")

if lift_low > threshold:
    print("✔ Гипотеза подтверждается.")
else:
    print("✘ Гипотеза НЕ подтверждается.")

Фактические значения:
CR TABLET:  0.30081
CR PHONE:   0.26940
Lift:       0.11660 (~11.66%)

Результаты Z-test:
Z-statistic: 0.7189
P-value:     0.236101

Доверительные интервалы:
CI CR TABLET: (0.21977, 0.38186)
CI CR PHONE:  (0.23613, 0.30267)
CI diff:      (-0.05620, 0.11903)
CI lift:      (-0.20861, 0.44182)

Порог гипотезы: 0.10 (10%)
✘ Гипотеза НЕ подтверждается.


Мы сравнили эффективность двух типов устройств — tablet и phone

**Результаты:**

*   CR TABLET: 30.08%
*   CR PHONE: 26.94%
*   Прирост CR (lift): +11.66%

То есть на уровне фактических данных пользователи планшетов действительно демонстрируют более высокую вероятность смены тарифа.

Однако статистическая проверка показала, что:

1. Различие между группами не является статистически значимым (p = 0.2361).

2. Доверительный интервал для разницы между CR включает отрицательные значения, что означает высокую неопределённость эффекта.

3. Доверительный интервал для лифта (от –20.86% до +44.18%) не достигает порога +10%, указанного в гипотезе.
Нижняя граница CI отрицательная, что говорит о том, что реальный эффект может быть как положительным, так и нулевым или даже отрицательным.

Это означает, что наблюдаемый рост конверсии у пользователей планшетов может быть обусловлен случайными колебаниями данных, и текущего объёма данных недостаточно, чтобы уверенно утверждать, что эффект стабильно превышает +10%.

**Итог: Гипотеза 4 не подтверждается на этапе CDA.**